In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier  

In [2]:
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
def entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-9))

In [4]:
def information_gain(y, y_left, y_right):
    H_parent = entropy(y)
    w_left = len(y_left) / len(y)
    w_right = len(y_right) / len(y)
    return H_parent - (w_left * entropy(y_left) + w_right * entropy(y_right))

In [5]:
class DecisionTreeScratch:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self.build_tree(X, y, depth=0)

    def build_tree(self, X, y, depth):
        if len(np.unique(y)) == 1 or depth >= self.max_depth:
            return np.bincount(y).argmax()

        best_feature, best_threshold, best_gain = None, None, -1

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for t in thresholds:
                left_idx = X[:, feature] <= t
                right_idx = X[:, feature] > t

                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue

                gain = information_gain(y, y[left_idx], y[right_idx])

                if gain > best_gain:
                    best_feature = feature
                    best_threshold = t
                    best_gain = gain

        if best_gain == -1:
            return np.bincount(y).argmax()

        left_idx = X[:, best_feature] <= best_threshold
        right_idx = X[:, best_feature] > best_threshold

        left = self.build_tree(X[left_idx], y[left_idx], depth + 1)
        right = self.build_tree(X[right_idx], y[right_idx], depth + 1)

        return {
            "feature": best_feature,
            "threshold": best_threshold,
            "left": left,
            "right": right
        }

    def predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree

        if x[tree["feature"]] <= tree["threshold"]:
            return self.predict_one(x, tree["left"])
        else:
            return self.predict_one(x, tree["right"])

    def predict(self, X):
        return np.array([self.predict_one(x, self.tree) for x in X])

In [6]:
model_scratch = DecisionTreeScratch(max_depth=3)
model_scratch.fit(X_train, y_train)

y_pred_scratch = model_scratch.predict(X_test)

acc_scratch = accuracy_score(y_test, y_pred_scratch)
print("Akurasi Decision Tree Scratch:", acc_scratch)

Akurasi Decision Tree Scratch: 1.0


In [7]:
model_sklearn = DecisionTreeClassifier(max_depth=3, random_state=42)
model_sklearn.fit(X_train, y_train)

y_pred_sklearn = model_sklearn.predict(X_test)

acc_sklearn = accuracy_score(y_test, y_pred_sklearn)
print("Akurasi Decision Tree Scikit-Learn:", acc_sklearn)

Akurasi Decision Tree Scikit-Learn: 1.0


In [9]:
print("Train Accuracy:", accuracy_score(y_train, model_scratch.predict(X_train)))
print("Test Accuracy :", accuracy_score(y_test, y_pred_scratch))

Train Accuracy: 0.9583333333333334
Test Accuracy : 1.0
